# 1. PyRIT Scan

`pyrit_scan` allows you to run automated security testing and red teaming attacks against AI systems using [scenarios](../scenarios/0_scenarios.ipynb) for strategies and [configuration](../setup/1_configuration.ipynb).

Note in this doc the ! prefaces all commands in the terminal so we can run in a Jupyter Notebook.

## Quick Start

For help:

In [ ]:
!pyrit_scan --help

Starting PyRIT...
usage: pyrit_scan [-h] [--config-file CONFIG_FILE] [--log-level LOG_LEVEL]
                  [--list-scenarios] [--list-initializers] [--list-targets]
                  [--initializers INITIALIZERS [INITIALIZERS ...]]
                  [--initialization-scripts INITIALIZATION_SCRIPTS [INITIALIZATION_SCRIPTS ...]]
                  [--strategies SCENARIO_STRATEGIES [SCENARIO_STRATEGIES ...]]
                  [--max-concurrency MAX_CONCURRENCY]
                  [--max-retries MAX_RETRIES] [--memory-labels MEMORY_LABELS]
                  [--dataset-names DATASET_NAMES [DATASET_NAMES ...]]
                  [--max-dataset-size MAX_DATASET_SIZE] [--target TARGET]
                  [scenario_name]

PyRIT Scanner - Run security scenarios against AI systems

Examples:
  # List available scenarios, initializers, and targets
  pyrit_scan --list-scenarios
  pyrit_scan --list-initializers
  pyrit_scan --list-targets --initializers target

  # Run a scenario with a target and i

### Discovery

List all available scenarios:

In [ ]:
!pyrit_scan --list-scenarios

Starting PyRIT...
Loading default configuration file: ./.pyrit/.pyrit_conf
Found default environment files: ['./.pyrit/.env']
Loaded environment file: ./.pyrit/.env

Available Scenarios:

  airt.content_harms
    Class: ContentHarms
    Description:
      Content Harms Scenario implementation for PyRIT. This scenario contains
      various harm-based checks that you can run to get a quick idea about
      model behavior with respect to certain harm categories.
    Aggregate Strategies:
      - all
    Available Strategies (7):
      hate, fairness, violence, sexual, harassment, misinformation, leakage
    Default Strategy: all
    Default Datasets (7, max 4 per dataset):
      airt_hate, airt_fairness, airt_violence, airt_sexual, airt_harassment,
      airt_misinformation, airt_leakage

  airt.cyber
    Class: Cyber
    Description:
      Cyber scenario implementation for PyRIT. This scenario tests how willing
      models are to exploit cybersecurity harms by generating malware. The
 

**Tip**: You can also discover user-defined scenarios by providing initialization scripts:

```shell
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_initializer.py
```

This will load your custom scenario definitions and include them in the list.

## Initializers

PyRITInitializers are how you can configure the CLI scanner. PyRIT includes several built-in initializers you can use with the `--initializers` flag.

The `--list-initializers` command shows all available initializers. Initializers are referenced by their filename (e.g., `target`, `objective_list`, `simple`) regardless of which subdirectory they're in.

List the available initializers using the --list-initializers flag.

In [ ]:
!pyrit_scan --list-initializers

Starting PyRIT...
Loading default configuration file: ./.pyrit/.pyrit_conf
Found default environment files: ['./.pyrit/.env']
Loaded environment file: ./.pyrit/.env

Available Initializers:

  airt
    Class: AIRTInitializer
    Name: AIRT Default Configuration
    Execution Order: 1
    Required Environment Variables:
      - AZURE_OPENAI_GPT4O_UNSAFE_CHAT_ENDPOINT
      - AZURE_OPENAI_GPT4O_UNSAFE_CHAT_MODEL
      - AZURE_OPENAI_GPT4O_UNSAFE_CHAT_ENDPOINT2
      - AZURE_OPENAI_GPT4O_UNSAFE_CHAT_MODEL2
      - AZURE_CONTENT_SAFETY_API_ENDPOINT
    Description:
      AI Red Team setup with Azure OpenAI converters, composite harm/objective
      scorers, and adversarial targets

  load_default_datasets
    Class: LoadDefaultDatasets
    Name: Default Dataset Loader for Scenarios
    Execution Order: 10
    Required Environment Variables: None
    Description:
      This configuration uses the DatasetLoader to load default datasets into
      memory. This will enable all scenarios to run

### Running Scenarios

You need a single scenario to run, you need two things:

1. A Scenario. Many are defined in `pyrit.scenario.scenarios`. But you can also define your own in initialization_scripts.
2. Initializers (which can be supplied via `--initializers` or `--initialization-scripts` or `initializers` section of config file (see [here](../../getting_started/pyrit_conf.md))). Scenarios often don't need many arguments, but they can be configured in different ways. And at the very least, most need an `objective_target` (the thing you're running a scan against) which you can configure by using the `--target` flag if your initializer registers targets (e.g. `target` initializer)
3. Scenario Strategies (optional). These are supplied by the `--scenario-strategies` flag and tell the scenario what to test, but they are always optional. Also note you can obtain these by running `--list-scenarios`

Basic usage will look something like:

```shell
pyrit_scan <scenario> --target <target_name> --initializers <initializer1> <initializer2> --scenario-strategies <strategy1> <strategy2>
```

You can also override scenario parameters directly from the CLI:

```shell
pyrit_scan <scenario> --max-concurrency 10 --max-retries 3 --memory-labels '{"experiment": "test1", "version": "v2"}'
```

Or concretely:

```shell
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers load_default_datasets target --scenario-strategies base64
```

Example with a basic configuration that runs the Foundry scenario against the objective target defined in the `target` initializer.

In [ ]:
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers load_default_datasets target --strategies base64

Starting PyRIT...
Loading default configuration file: ./.pyrit/.pyrit_conf
Found default environment files: ['./.pyrit/.env']
Loaded environment file: ./.pyrit/.env
Running 2 initializer(s)...

Running scenario: foundry.red_team_agent

                                  📊 SCENARIO RESULTS: RedTeamAgent                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: RedTeamAgent
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        RedTeamAgent is a preconfigured scenario that automatically generates multiple AtomicAttack instances based on
        the specified attack strategies. It supports both single-turn attacks (with various converters) and multi-turn
        attacks (Crescendo, RedTeaming), making it easy to quickly test a target against multiple attack vectors. The
        scenario can expand difficulty levels (EASY, MODER


Loading datasets - this can take a few minutes: 100%|██████████| 58/58 [00:00<00:00, 65.13dataset/s]

Executing RedTeamAgent: 100%|██████████| 2/2 [00:15<00:00,  7.59s/attack]
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001A51ECDEB50>


Or with all options and multiple initializers and multiple strategies:

```shell
pyrit_scan foundry.red_team_agent --target openai_chat --initializers load_default_datasets target --strategies easy crescendo
```

You can also override scenario execution parameters:

```shell
# Override concurrency and retry settings
pyrit_scan foundry.red_team_agent --target openai_chat --initializers load_default_datasets target --max-concurrency 10 --max-retries 3

# Add custom memory labels for tracking (must be valid JSON)
pyrit_scan foundry.red_team_agent --target openai_chat --initializers load_default_datasets target --memory-labels '{"experiment": "test1", "version": "v2", "researcher": "alice"}'
```

Available CLI parameter overrides:
- `--max-concurrency <int>`: Maximum number of concurrent attack executions
- `--max-retries <int>`: Maximum number of automatic retries if the scenario raises an exception
- `--memory-labels <json>`: Additional labels to apply to all attack runs (must be a JSON string with string keys and values)

You can also use custom initialization scripts by passing file paths. It is relative to your current working directory, but to avoid confusion, full paths are always better:

```shell
pyrit_scan garak.encoding --initialization-scripts ./my_custom_config.py
```

#### Using Custom Scenarios

You can define your own scenarios in initialization scripts. The CLI will automatically discover any `Scenario` subclasses and make them available:


In [ ]:
# my_custom_scenarios.py

from pyrit.common import apply_defaults
from pyrit.prompt_target.openai.openai_chat_target import OpenAIChatTarget
from pyrit.scenario import DatasetConfiguration, Scenario, ScenarioStrategy
from pyrit.score import SelfAskRefusalScorer, TrueFalseInverterScorer
from pyrit.setup import initialize_pyrit_async


class MyCustomStrategy(ScenarioStrategy):
    """Strategies for my custom scenario."""

    ALL = ("all", {"all"})
    Strategy1 = ("strategy1", set[str]())
    Strategy2 = ("strategy2", set[str]())


class MyCustomScenario(Scenario):
    """My custom scenario that does XYZ."""

    @classmethod
    def get_strategy_class(cls):
        return MyCustomStrategy

    @classmethod
    def get_default_strategy(cls):
        return MyCustomStrategy.ALL

    @classmethod
    def default_dataset_config(cls) -> DatasetConfiguration:
        # Return default dataset configuration for this scenario
        return DatasetConfiguration(dataset_names=["harmbench"])

    @apply_defaults
    def __init__(self, *, scenario_result_id=None, **kwargs):
        # Scenario-specific configuration only - no runtime parameters
        super().__init__(
            name="My Custom Scenario",
            version=1,
            objective_scorer=TrueFalseInverterScorer(scorer=SelfAskRefusalScorer(chat_target=OpenAIChatTarget())),
            strategy_class=MyCustomStrategy,
            scenario_result_id=scenario_result_id,
        )
        # ... your scenario-specific initialization code

    async def _get_atomic_attacks_async(self):
        # Build and return your atomic attacks based on self._scenario_composites
        # Example: create attacks for each strategy composite
        return []


await initialize_pyrit_async(memory_db_type="InMemory")  # type: ignore
MyCustomScenario()

Found default environment files: ['./.pyrit/.env']
Loaded environment file: ./.pyrit/.env


Then discover and run it:

```shell
# List to see it's available
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_scenarios.py

# Run it with parameter overrides
pyrit_scan my_custom_scenario --initialization-scripts ./my_custom_scenarios.py --max-concurrency 10
```

The scenario name is automatically converted from the class name (e.g., `MyCustomScenario` becomes `my_custom_scenario`).